# Mapping Urban Heat at Higher Resolution: Machine Learning–Based Downscaling of ECOSTRESS LST from 70 m to 10 m in Python

by Aycha Tammour

In this notebook, i demostrate the process of downscaling ECOSTRESS Land Surface Temperature (LST) data from its native 70m resolution to 10m. I follow the approach presented in a NASA ARSET training module using Google Earth Engine, but implement it in Python using open-source libraries and data.



## How the downscaling work?

The idea is simple, we know that the low resolution ECOSTRESS LST data is correlated with the higher resolution reflectance data. This correlation means that if we have a sample of the LST data and reflectance data for the same AOI, we can train a machine learning model to learn the relationship between the two. Once we have a trained model, we can use it to predict the LST values at the higher resolution using the reflectance data as input.


```{=html}
<div class="mermaid">
flowchart LR
    A[Collect training data]
    A --> B[Target: LST at 70 m]
    A --> C[Features: Sentinel-2 Reflectance and DEM at 70 m]
    B --> D[Train Random Forest model]
    C --> D
    D --> E[Evaluate model performance]
    D --> F[Collect predictor data at 10 m resolution: Sentinel-2 Reflectance and DEM]
    F --> G[Predict LST at 10 m Resolution]
</div>
```

## Location and Dates

### Area of Interest: Toronto, Canada

We will focus on Toronto...

In the summer months...

In [89]:
import geopandas as gpd
from shapely.geometry import box

aoi_name = ["london", "toronto", "vancouver", "cairo"]
target_crs = ["EPSG:32630", "EPSG:32617", "EPSG:32610", "EPSG:32636"] # UTM 30N, 17N, 10N, 36N
bbox = [(-0.51, 51.28, 0.33, 51.75), # min_lon, min_lat, max_lon, max_lat
        (-79.91, 43.51, -79, 43.93), 
        (-123.30, 49.15, -122.90, 49.35),
        (31.10, 29.90, 31.50, 30.20)] # extent around london, toronto, vancouver, cairo
timezone = ["Europe/London", "America/Toronto", "America/Vancouver", "Africa/Cairo"]
start_date = ['2025-08-01', '2025-06-15', '2025-06-15', '2025-05-01']
end_date = ['2025-08-15', '2025-08-15', '2025-08-15', '2025-09-30']


locations_gdf = gpd.GeoDataFrame(
    {'AOI Name': aoi_name,
     'Target CRS': target_crs,
     'Timezone': timezone,
     'Start Date': start_date,
     'End Date': end_date,
    'geometry': gpd.GeoSeries([box(*b) for b in bbox])},
    crs='EPSG:4326'
)

# Save GeoDataFrame to GeoJSON
locations_gdf.to_file(f"locations.geojson", driver="GeoJSON")

In [90]:
locations_gdf

,AOI Name,Target CRS,Timezone,Start Date,End Date,geometry
0,london,EPSG:32630,Europe/London,2025-08-01,2025-08-15,"POLYGON ((0.33 51.28, 0.33 51.75, -0.51 51.75,..."
1,toronto,EPSG:32617,America/Toronto,2025-06-15,2025-08-15,"POLYGON ((-79 43.51, -79 43.93, -79.91 43.93, ..."
2,vancouver,EPSG:32610,America/Vancouver,2025-06-15,2025-08-15,"POLYGON ((-122.9 49.15, -122.9 49.35, -123.3 4..."
3,cairo,EPSG:32636,Africa/Cairo,2025-05-01,2025-09-30,"POLYGON ((31.5 29.9, 31.5 30.2, 31.1 30.2, 31...."


In [91]:
location_info = locations_gdf.loc[locations_gdf['AOI Name'] == 'toronto']

aoi_name = location_info['AOI Name'].values[0]
target_crs = location_info['Target CRS'].values[0]
timezone = location_info['Timezone'].values[0]
start_date = location_info['Start Date'].values[0]
end_date = location_info['End Date'].values[0]
(west, south, east, north) = location_info['geometry'].values[0].bounds
bbox = (west, south, east, north)

print("Extracted Location Information:")
print("-" * 30)
print(f"AOI Name: {aoi_name}")
print(f"Target CRS: {target_crs}")
print(f"Timezone: {timezone}")
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"Bounding Box: (west: {west}, south: {south}, east: {east}, north: {north})")


Extracted Location Information:
------------------------------
AOI Name: toronto
Target CRS: EPSG:32617
Timezone: America/Toronto
Start Date: 2025-06-15
End Date: 2025-08-15
Bounding Box: (west: -79.91, south: 43.51, east: -79.0, north: 43.93)


In [92]:
location_info.explore()

# Training Data

The data used in this notebook includes

- ECOSTRESS LST: 70m resolution from NASA's ECOSTRESS mission
    this is the main variable we want to downscale...
- Sentinel-2: 70m, and 10m resolution from ESA's Sentinel-2 mission
    first download at 70m to train the model, then at 10m to predict the downscaled LST
- DEM: 30m resolution from NASA's SRTM mission
    elevation data can be useful predictors for LST, so we will include it as an additional variable in our model

For the ECOSTRESS LST data, we will use the `earthaccess` library to search for and download the relevant granules that intersect with our AOI and fall within our specified date range. For the Sentinel-2 data and the DEM data, we can use the `pystac_client`, `odc-stac`, and `Xarray` to search for and download the relevant data.
Unfortunately, ECOSTRESS data is only available via STAC if we are in a cloud environment (e.g. Google Colab) and not on a local machine. So for this notebook, I will demonstrate the process of searching for and downloading the ECOSTRESS LST data using `earthaccess`, and then we can use the downloaded data to train our model and perform the downscaling.

## LST Data

To get the ECOSTRESS LST data, we will use the `earthaccess` library to search for and download the relevant granules that intersect with our AOI and fall within our specified date range. This requires setting up an Earthdata account and configuring the `earthaccess` library with your credentials. Once you have that set up, you can use the following code to search for and download the ECOSTRESS LST data for our AOI and date range.

In [93]:
# Authenticate
import earthaccess

auth = earthaccess.login(persist=True)
print(auth.authenticated)

True


Now let's search for the ECOSTRESS LST data using `earthaccess`. We will specify our AOI, date range, and the collection we want to search in. Then we can download the relevant granules to our local machine.

In [94]:
import warnings
warnings.filterwarnings('ignore')

In [95]:
# Search
import pandas as pd

lst_granules = earthaccess.search_data(
    short_name="ECO_L2T_LSTE",
    version="002",
    bounding_box=(west, south, east, north),
    temporal=(start_date, end_date),
)
print(f"Found {len(lst_granules)} LST granules in the AOI and date range.")

Found 87 LST granules in the AOI and date range.


We could do a bit more filtering here to only download the granules that have good quality data. To do this, we can loop through the LST granules and check for the following conditions before downloading:
- time range between noon and 6 PM (this can be checked on the fly before saving the data, by looking at the timestamp of each granule)
- stream the data (via earthaccess' fsspec) and check the % of valid pixels (not nulls) in the LST band
- if the % of valid pixels is > 75% then save it to disk and download the associated water and QC mask files as well.

This approach runs efficiently and helps us avoid downloading and processing granules that have a lot of missing data, which can save time and storage space. The resulting directory in this case contains only 21 files (data, water masks, QC masks) and is approximately 72 MB in size.

The script for this process is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/keep_valid_granules.py)

In [96]:
from granules_filter import GranuleFilter

gfilter = GranuleFilter(aoi_name=aoi_name,
                        timezone=timezone,
                        valid_pixel_threshold=0.7)

In [110]:
day_or_night = "day" #"night"  # or "day"
good_granules = gfilter.keep_valid_granules(lst_granules, 
                                            day_or_night=day_or_night)


Skipping (outside time window): 2025-06-16T16:53:09.820Z
Skipping (outside time window): 2025-06-16T16:53:09.820Z
Skipping (outside time window): 2025-06-16T16:54:01.790Z
Skipping (outside time window): 2025-06-16T16:54:01.790Z
Skipping (outside time window): 2025-06-19T16:03:53.852Z
Skipping (outside time window): 2025-06-19T16:04:45.822Z
Skipping (outside time window): 2025-06-19T16:04:45.822Z
Skipping (outside time window): 2025-06-20T15:15:31.157Z
Skipping (outside time window): 2025-06-20T15:15:31.157Z
Skipping (outside time window): 2025-06-20T15:16:23.127Z
Skipping (outside time window): 2025-06-20T15:16:23.127Z
Skipping (outside time window): 2025-06-23T14:27:14.220Z
Skipping (outside time window): 2025-06-23T14:27:14.220Z
Skipping (outside time window): 2025-06-24T08:46:47.269Z
Skipping (outside time window): 2025-06-24T08:46:47.269Z
Skipping (outside time window): 2025-06-24T13:38:15.089Z
Skipping (outside time window): 2025-06-24T13:38:15.089Z
Skipping (outside time window):

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

08_17TPJ_20250803T220909_0713_01_LST.tif  valid: 75.6%


QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

Skipping (outside time window): 2025-08-04T21:19:43.254Z
Skipping (outside time window): 2025-08-04T21:19:43.254Z
Skipping (outside time window): 2025-08-04T21:20:35.223Z
Skipping (outside time window): 2025-08-04T21:20:35.223Z
Skipping (outside time window): 2025-08-05T15:39:01.961Z
Skipping (outside time window): 2025-08-07T20:30:04.882Z
Skipping (outside time window): 2025-08-07T20:30:56.849Z
Skipping (outside time window): 2025-08-07T20:30:56.849Z
Skipping (outside time window): 2025-08-08T19:41:48.108Z
Skipping (outside time window): 2025-08-08T19:42:40.078Z
Skipping (outside time window): 2025-08-11T18:52:30.008Z
Skipping (outside time window): 2025-08-11T18:52:30.008Z
Skipping (outside time window): 2025-08-15T17:14:00.598Z
Skipping (outside time window): 2025-08-15T17:14:52.568Z
Skipping (outside time window): 2025-08-15T17:14:52.568Z


In [111]:
print(f"\nKept {len(good_granules)} clean granules for {day_or_night} LST.")


Kept 2 clean granules for day LST.


In [112]:
good_granules[1]

{'granule': Collection: {'ShortName': 'ECO_L2T_LSTE', 'Version': '002'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -79.76796003845342, 'EastBoundingCoordinate': -78.37385429483595, 'NorthBoundingCoordinate': 44.24655093448751, 'SouthBoundingCoordinate': 43.23597190336682}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-08-03T22:09:09.536Z', 'EndingDateTime': '2025-08-03T22:10:01.506Z'}}
 Size(MB): 12.78
 Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01_water.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01_cloud.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/ECO_L2T_LSTE.002/ECO

In [113]:
import json

def serialize_entry(entry):
    return {
        'granule': dict(entry['granule']),   # CMR JSON underneath, now plain dict
        'lst_file': entry['lst_file'],
        'water_file': entry['water_file'],
        'qc_file': entry['qc_file'],
        'cloud_file': entry['cloud_file']
    }

serializable = [serialize_entry(e) for e in good_granules]

with open(f'{aoi_name}_{day_or_night}_granules.json', 'w') as f:
    json.dump(serializable, f, indent=2, default=str)  # default=str catches any stray datetimes

Le's take a look at the output of the script

### LST Data Processing and Stacking

<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
<div class="mermaid">
flowchart LR
    A[Search] --> B[Filter\ndate + cloud]
    B --> C[Filter\ntime + valid pixels]
    C --> D[Download]
    D --> E[QC mask\nwater mask\nclip to AOI]
    E --> F[Stack]
</div>
<script>mermaid.initialize({startOnLoad:true})</script>

Before we use the data, we need to perform some preprocessing steps. This includes QC masking to keep only good quality pixels. We also apply the water mask file to keep only land pixels. Finally we clip the data to our AOI.

The scrip is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/preprocess_lst.py).
Let's apply it to the LST data we downloaded and see the results.

We need to pause here for a moment to check the data we have. The AOI we chose is not covered by one granule but rather by two tiles. We can infer this from the file names which include `17TNJ` and `17TPJ`. 
`17T` is just the UTM zone. `NJ` and `PJ` are the tile indentifiers in the [Military Grid Reference System](https://en.wikipedia.org/wiki/Military_Grid_Reference_System) (MGRS) tile codes. It's the same tiling system used by Sentinel-2 and HLS. 

In [19]:
lst_file = "./toronto_ecostress_data/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01_lst.tif"
cloud_file = "./toronto_ecostress_data/ECOv002_L2T_LSTE_40129_008_17TPJ_20250803T220909_0713_01_cloud.tif"

In [20]:
import rioxarray
import numpy as np

lst_raw = rioxarray.open_rasterio(lst_file, masked=True).squeeze()
cloud_raw = rioxarray.open_rasterio(cloud_file, masked=True).squeeze()
print(np.unique(cloud_raw.values[~np.isnan(cloud_raw.values)]))

[0. 1.]


Mask the data using the QC and water masks, and then clip it to our AOI.

- For the QC mask, we will keep pixels with values of 00 (good quality), and 01 (acceeptable quality).

- For the water mask, we will keep pixels with values of 0 (land) and mask out pixels with values of 1 (water).

The full preprocessing script is available on GitHub [here](https://github.com/astroAycha/ecostress-lst-downscaling/blob/main/preprocess_lst.py).

Once we have proccessed the LST data, we can stack the granules and create a mean composite for our AOI. This is the 70 m resolution LST data that we will be one of the entries used to train our model.

In [114]:
from collections import defaultdict

from collections import defaultdict
import json

def group_granules_by_tile(json_file_path):
    with open(json_file_path) as f:
        loaded = json.load(f)

    tiles = defaultdict(list)
    for entry in loaded:
        # normalize \ and / then take just the filename
        filename = entry['lst_file'].replace('\\', '/').split('/')[-1]
        tile_id = filename.split('_')[-5]
        tiles[tile_id].append(entry)

    return dict(tiles)

In [115]:
# check the function works
tiles = group_granules_by_tile(f'{aoi_name}_{day_or_night}_granules.json')
print(list(tiles.keys()))

['17TNJ', '17TPJ']


In [116]:
from build_composite import build_reference_grid

resolution = 70  # ECOSTRESS native resolution, meters
transform, shape = build_reference_grid(bbox, target_crs, resolution)

In [117]:
from build_composite import build_composite

max_cloud_cover = 10

tile_composites = {}
for tile_id, entries in tiles.items():
    print(f"Building composite for tile {tile_id} ({len(entries)} granules)")
    composite = build_composite(entries, 
                                bbox, 
                                target_crs, 
                                transform, shape,
                                max_cloud_cover=max_cloud_cover)
    if composite is not None:
        tile_composites[tile_id] = composite

Building composite for tile 17TNJ (1 granules)
  QC mask applied >>> 82.8% pixels retained
  Cloud cover in AOI: 4.8%
Building composite for tile 17TPJ (1 granules)
  QC mask applied >>> 75.6% pixels retained
  Cloud cover in AOI: 2.5%


In [118]:
from build_composite import merge_tiles
# lst_mosaic = merge_tiles(tile_composites, target_crs, feather_px=15)
lst_mosaic = merge_tiles(tile_composites, 
                        target_crs,
                        feather_px=15)

We can now build our composite LST. We will build two and then combine them in a way that allows us to deal with the overlap of the two tiles. If we just simply take the mean, the two tiles create edges in the final composite so we will take the mean of the area wheret the two composites to create a smoother final product.

In [119]:
# save the mosaic
lst_mosaic.rio.to_raster(f"./products/{aoi_name}_{day_or_night}_lst_mosaic_70m.tif", 
                         driver="GTiff",
                         dtype="float32")

In [21]:
import numpy as np

print(lst_mosaic.notnull().sum())

<xarray.DataArray ()> Size: 8B
array(266708)
Coordinates:
    band         int64 8B 1
    spatial_ref  int64 8B 0


In [22]:
lst_mosaic

<xarray.DataArray (y: 484, x: 559)> Size: 2MB
array([[301.54998779, 301.70999146, 301.89001465, ..., 306.36999512,
        306.69000244, 307.20001221],
       [         nan, 301.85998535, 302.3500061 , ..., 306.40997314,
        306.45001221, 306.67999268],
       [         nan,          nan, 301.82000732, ..., 306.40997314,
        306.45001221, 306.67999268],
       ...,
       [306.42999268, 309.64001465, 309.64001465, ..., 304.16000366,
        303.85998535, 304.15002441],
       [306.77999878, 309.54000854, 306.72000122, ..., 304.16000366,
        303.85998535, 304.15002441],
       [308.73999023, 307.22000122, 309.57998657, ..., 303.85998535,
        303.77999878, 303.88000488]], shape=(484, 559))
Coordinates:
  * x            (x) float64 4kB 3.166e+05 3.167e+05 ... 3.556e+05 3.556e+05
  * y            (y) float64 4kB 3.342e+06 3.342e+06 ... 3.309e+06 3.309e+06
    band         int64 8B 1
    spatial_ref  int64 8B 0
Attributes:
    _FillValue:  nan

Let's make a plot of the composite LST to see what it looks like

In [23]:
import hvplot.xarray

lst_mosaic.rio.reproject("EPSG:4326").hvplot.image(x="x", 
                                                   y="y", 
                                                   rasterize=True, 
                                                   cmap="coolwarm", 
                                                   title="LST", 
                                                   frame_width=600, 
                                                   frame_height=400,
                                                   geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'e78cd04a-f93e-47b5-8664-b6d91a4f84e4': {'version…

Now that we are done with the hard part, let's move on to the next step which is to get the rest of the training data: Sentinel-2 reflectance data and the DEM data in both 70 and 10 m resolution.

## Sentinel-2 data:

We can now use the `pystac_client` and `odc-stac` libraries to search for and download the Sentinel-2 data for our AOI and date range. We will need to specify the collection we want to search in (e.g. "sentinel-2-l2a") and the bands we want to download (e.g. B02, B03, B04, B08). We can also specify the resolution we want to download at (e.g. 10m or 20m).

I went through this process in detail in two previous notebooks [here]() and [here](), so I won't repeat the details again. A lot of what was done for the ECOSTRESS LST data in terms of searching, and aligning the data to the same grid is now done using the `odc-stac` library, which makes it much easier to work with the data in Python.

In [24]:
import pystac_client
import odc.stac

In [25]:
s2_catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")

s2_items = s2_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=f"{start_date}/{end_date}",
).item_collection()

print(f"Found {len(s2_items)} Sentinel-2 items in the AOI and date range.")

Found 76 Sentinel-2 items in the AOI and date range.


In [26]:
s2_70_ds = odc.stac.load(
    s2_items,
    bbox=bbox,
    bands=["red", "green", "blue", "nir"],
    resolution=70, # <- use the same resolution as the LST data
    chunks={},
    groupby="solar_day",
    crs="EPSG:32617",
    resampling="bilinear",
    query={"eo:cloud_cover": {"lt": 10}}
)

In [27]:
s2_70_ds

<xarray.Dataset> Size: 754MB
Dimensions:      (y: 1214, x: 1195, time: 65)
Coordinates:
  * y            (y) float64 10kB 1.373e+07 1.373e+07 ... 1.365e+07 1.365e+07
  * x            (x) float64 10kB 7.48e+06 7.48e+06 ... 7.564e+06 7.564e+06
    spatial_ref  int32 4B 32617
  * time         (time) datetime64[ns] 520B 2025-05-02T08:42:13.992000 ... 20...
Data variables:
    red          (time, y, x) uint16 189MB dask.array<chunksize=(1, 1214, 1195), meta=np.ndarray>
    green        (time, y, x) uint16 189MB dask.array<chunksize=(1, 1214, 1195), meta=np.ndarray>
    blue         (time, y, x) uint16 189MB dask.array<chunksize=(1, 1214, 1195), meta=np.ndarray>
    nir          (time, y, x) uint16 189MB dask.array<chunksize=(1, 1214, 1195), meta=np.ndarray>

In [28]:
from rasterio.enums import Resampling

# force S2 onto exactly the same grid as LST
s2_70_ds_matched = s2_70_ds.rio.reproject_match(lst_mosaic, 
                                                resampling=Resampling.average
                                                )

# verify
print("LST shape:", lst_mosaic.shape)
print("S2 shape: ", s2_70_ds_matched["blue"].mean(dim="time").shape)
print("LST bounds:", lst_mosaic.rio.bounds())
print("S2 bounds: ", s2_70_ds_matched.rio.bounds())

LST shape: (484, 559)
S2 shape:  (484, 559)
LST bounds: (316548.9111847761, 3308593.1086698733, 355678.9111847761, 3342473.1086698733)
S2 bounds:  (316548.9111847761, 3308593.1086698733, 355678.9111847761, 3342473.1086698733)


In [29]:
import numpy as np

# squeeze time dim and take mean composite 
blue_70 = s2_70_ds_matched["blue"].mean(dim="time").values.astype(float)
green_70 = s2_70_ds_matched["green"].mean(dim="time").values.astype(float)
red_70 = s2_70_ds_matched["red"].mean(dim="time").values.astype(float)
nir_70 = s2_70_ds_matched["nir"].mean(dim="time").values.astype(float)

# DEM data

We do the same for the elevation data. Elevation might not have a big impact for our AOI since Toronto is relatively flat, but it is better to include it as a predictor in our model just in case it does have some influence on the LST values and to be consistent with the approach used in the NASA ARSET training module.

In [30]:
dem_catalog = pystac_client.Client.open(
    "https://earth-search.aws.element84.com/v1"
)

dem_items = dem_catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=bbox
).item_collection()

print(f"Found {len(dem_items)} DEM items")

Found 2 DEM items


In [31]:
import odc.stac

dem_70_ds = odc.stac.load(
    dem_items,
    bbox=bbox,
    bands=["data"],
    resolution=70, # <- use the same resolution as the LST data
    groupby="solar_day",
    chunks={},
    crs="EPSG:32617",
    resampling="bilinear"
)

In [32]:
dem_70_ds

<xarray.Dataset> Size: 6MB
Dimensions:      (y: 1214, x: 1195, time: 1)
Coordinates:
  * y            (y) float64 10kB 1.373e+07 1.373e+07 ... 1.365e+07 1.365e+07
  * x            (x) float64 10kB 7.48e+06 7.48e+06 ... 7.564e+06 7.564e+06
    spatial_ref  int32 4B 32617
  * time         (time) datetime64[ns] 8B 2021-04-22
Data variables:
    data         (time, y, x) float32 6MB dask.array<chunksize=(1, 1214, 1195), meta=np.ndarray>

In [33]:
# force DEM onto exactly the same grid as LST
dem_70_matched = dem_70_ds.rio.reproject_match(lst_mosaic, 
                                                resampling=Resampling.average
                                                )
# squeeze dem
dem_70 = dem_70_matched["data"].squeeze().values.astype(float)

# verify
print("LST shape:", lst_mosaic.shape)
print("DEM shape: ", dem_70.shape)
print("LST bounds:", lst_mosaic.rio.bounds())
print("DEM bounds: ", dem_70_matched.rio.bounds())

LST shape: (484, 559)
DEM shape:  (484, 559)
LST bounds: (316548.9111847761, 3308593.1086698733, 355678.9111847761, 3342473.1086698733)
DEM bounds:  (316548.9111847761, 3308593.1086698733, 355678.9111847761, 3342473.1086698733)


In [34]:
# derive land_mask from the water-masked mosaic
# pixels that survived clip_and_water_mask_lst are land; water pixels are NaN
land_mask = np.isfinite(lst_mosaic.values)
print(f"Land pixels in mosaic: {land_mask.sum()} / {land_mask.size}")

Land pixels in mosaic: 266708 / 270556


In [35]:
print("lst:       ", lst_mosaic.shape)
print("land_mask: ", land_mask.shape)
print("blue:      ", blue_70.shape)
print("green:     ", green_70.shape)
print("red:       ", red_70.shape)
print("nir:       ", nir_70.shape)
print("elev:      ", dem_70.shape)

lst:        (484, 559)
land_mask:  (484, 559)
blue:       (484, 559)
green:      (484, 559)
red:        (484, 559)
nir:        (484, 559)
elev:       (484, 559)


In [36]:
lst = lst_mosaic.values.astype(float)  # extract numpy array from xarray

# combined mask — land + valid LST + valid S2 + valid DEM
mask = (
    land_mask            &   # water mask
    np.isfinite(lst)     &   # valid LST
    (lst > 0)            &   # no zero LST
    np.isfinite(blue_70)      &   # valid S2
    np.isfinite(green_70)      &
    np.isfinite(red_70)      &
    np.isfinite(nir_70)      &
    np.isfinite(dem_70)        # valid DEM
)

print(f"Valid training pixels after all masks: {mask.sum()}")

Valid training pixels after all masks: 266577


In [37]:
# flatten
def flat(arr):
    return arr[mask].ravel()

X_full = np.column_stack([flat(blue_70), 
                          flat(green_70), 
                          flat(red_70), 
                          flat(nir_70),
                          flat(dem_70)
                          ])
y_full = flat(lst)

print(f"X shape: {X_full.shape}")
print(f"y range: {y_full.min():.1f} — {y_full.max():.1f} K")

X shape: (266577, 5)
y range: 299.0 — 311.2 K


In [38]:
# # don't subsample — use all valid pixels
# X_train, y_train = X_full, y_full
# print(f"Training on all {X_train.shape[0]} valid pixels")

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_full, 
                                                    y_full, 
                                                    test_size=0.2, 
                                                    random_state=11)
print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

Training samples: 213261, Testing samples: 53316


In [40]:
# # S2 nodata is often 0 — treat 0 as invalid
# valid_s2 = (
#     np.isfinite(blue_70) & (blue_70 > 0) &
#     np.isfinite(green_70) & (green_70 > 0) &
#     np.isfinite(red_70) & (red_70 > 0) &
#     np.isfinite(nir_70) & (nir_70 > 0)
# )
# print(f"Valid S2 pixels with >0 check: {valid_s2.sum()}")

In [41]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=50, 
                           random_state=79, 
                           n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,50
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [42]:
from sklearn.metrics import mean_absolute_error as mae

y_pred = rf.predict(X_test)
error = mae(y_test, y_pred)
print(f"Mean Absolute Error: {error:.2f} K")

Mean Absolute Error: 0.57 K


In [2]:
#TODO: Add feature importance plot, partial dependence plots, and error scatter plot

# Prediction and evaluation

For the prediction step, we will use a machine learning model to learn the relationship between the low resolution LST data and the high resolution reflectance data. We can use a simple regression model for this, such as a Random Forest or a Gradient Boosting Regressor. We will train the model using the 10m resolution data, and then use it to predict the LST values at 70m resolution using the reflectance data as input.

## to 10m

In [43]:
# load S2 at 10m
s2_10_ds = odc.stac.load(
    s2_items,
    bbox=bbox,
    bands=["blue", "green", "red", "nir"],
    resolution=10,
    chunks={},
    groupby="solar_day",
    crs="EPSG:32617",
    resampling="bilinear",
    query={"eo:cloud_cover": {"lt": 10}}
)


In [44]:
blue_10 = s2_10_ds["blue"].mean(dim="time").values.astype(float)
green_10 = s2_10_ds["green"].mean(dim="time").values.astype(float)
red_10 = s2_10_ds["red"].mean(dim="time").values.astype(float)
nir_10 = s2_10_ds["nir"].mean(dim="time").values.astype(float)

In [45]:
# reproject DEM to 10m
dem_10_matched = dem_70_ds["data"].rio.reproject_match(s2_10_ds,
                                                resampling=Resampling.bilinear)

dem_10 = dem_10_matched.squeeze().values.astype(float)

In [46]:

# valid mask at 10m
mask_10 = (
    np.isfinite(blue_10)  & (blue_10 > 0)  &
    np.isfinite(green_10) & (green_10 > 0) &
    np.isfinite(red_10)   & (red_10 > 0)   &
    np.isfinite(nir_10)   & (nir_10 > 0)   &
    np.isfinite(dem_10)
)

In [47]:
X_pred_10 = np.column_stack([blue_10[mask_10], 
                             green_10[mask_10], 
                             red_10[mask_10], 
                             nir_10[mask_10],
                             dem_10[mask_10]])

print(f"Predicting over {X_pred_10.shape[0]} pixels at 10m...")
y_pred_10 = rf.predict(X_pred_10)

Predicting over 69750081 pixels at 10m...


In [48]:
# reconstruct 2D
lst_10m = np.full(mask_10.shape, np.nan)
lst_10m[mask_10] = y_pred_10

print(f"LST 10m shape: {lst_10m.shape}")
print(f"Value range:   {np.nanmin(lst_10m):.1f} — {np.nanmax(lst_10m):.1f} K")

LST 10m shape: (8491, 8361)
Value range:   299.7 — 310.7 K


In [49]:
import xarray as xr

# wrap 10m prediction as DataArray
lst_10m_da = xr.DataArray(
    lst_10m,
    dims=["y", "x"],
    coords={"y": s2_10_ds.y, "x": s2_10_ds.x}
).rio.write_crs("EPSG:32617")

# aggregate prediction back to 70m
lst_pred_agg = lst_10m_da.rio.reproject_match(
    lst_mosaic, resampling=Resampling.average
)

# residual = original 70m LST - aggregated prediction
residual_70m = lst_mosaic - lst_pred_agg

# resample residual to 10m (bilinear = smooth surface)
residual_10m = residual_70m.rio.reproject_match(
    lst_10m_da, resampling=Resampling.bilinear
)

# final bias-corrected 10m LST
lst_final = lst_10m_da + residual_10m

print(f"Final LST 10m range: {float(lst_final.min()):.1f} — {float(lst_final.max()):.1f} K")

Final LST 10m range: 297.8 — 313.3 K


In [50]:
print(f"Residual mean: {float(residual_70m.mean()):.2f} K")
print(f"Residual std:  {float(residual_70m.std()):.2f} K")
print(f"Residual max abs: {float(abs(residual_70m).max()):.2f} K")

Residual mean: -0.03 K
Residual std:  0.80 K
Residual max abs: 5.90 K


In [51]:
# convert to Celsius for easier reading
lst_70m_C = lst_mosaic - 273.15
lst_10m_C = lst_final - 273.15

In [52]:
lst_70m_C.rio.to_raster(f"./products/{aoi_name}_lst_70m_C.tif",
                         driver="GTiff",
                            dtype="float32")

In [53]:
lst_10m_C.rio.to_raster(f"./products/{aoi_name}_lst_10m_C.tif",
                         driver="GTiff",
                            dtype="float32")

In [51]:
lst_70m_C.hvplot.image(x="x", 
                 y="y", 
                 rasterize=True, 
                 cmap="coolwarm", 
                 title="Downscaled LST at 70m (°C)", 
                 frame_width=600, 
                 frame_height=400,
                 geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'52a1f1d6-7b18-4b82-9df3-cc84ad61a7d6': {'version…

In [52]:
lst_10m_C.hvplot.image(x="x", 
                 y="y", 
                 rasterize=True, 
                 cmap="coolwarm", 
                 title="Downscaled LST at 10m (°C)", 
                 frame_width=600, 
                 frame_height=400,
                 geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'1ca54fd9-4aaf-4374-b614-297aa98ea047': {'version…

### Acknowledgement:

This exercise is based on the NASA ARSET training module on downscaling ECOSTRESS LST data using Google Earth Engine. The original training module can be found [here](https://www.earthdata.nasa.gov/learn/trainings/introduction-thermal-remote-sensing-applications-urban-heat-island-mapping). I have adapted the approach presented in the training module to work in Python using open-source libraries and data.

## Useful Links

- [Introduction to Thermal Remote Sensing and Applications in Urban Heat Island Mapping](https://www.earthdata.nasa.gov/learn/trainings/introduction-thermal-remote-sensing-applications-urban-heat-island-mapping)
- [Introduction to `earthaccess`](https://github.com/nasa/LPDAAC-Data-Resources/blob/main/python/tutorials/earthaccess_introduction.ipynb)
- [ECOSTRESS Data Resouces on GitHub](https://github.com/nasa/ECOSTRESS-Data-Resources)
- [Working with ECOSTRESS Tiled Data](https://github.com/nasa/ECOSTRESS-Data-Resources/blob/20dfefbfb793d924aa6b222ddc4c9464e66ff4a6/python/tutorials/Working_with_ECOSTRESS_Tiled_data.ipynb)

In [120]:
import datetime

print(f"Last updated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Last updated: 2026-06-23 20:58:25
